# Case Study 2 : Putting NewsAPI Data into the Cloud

Due Date: 11/24/2026, **BEFORE the beginning of class at 12:00pm EST**

## **NOTE: There are *always* last minute issues submitting the case studies.  DO NOT WAIT UNTIL THE LAST MINUTE!**

<a title="Fabián Alexis [CC BY-SA (https://creativecommons.org/licenses/by-sa/3.0)]" href="https://commons.wikimedia.org/wiki/File:Antu_mongodb.svg"><img width="512" alt="Antu mongodb" src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/f9/Antu_mongodb.svg/512px-Antu_mongodb.svg.png"></a>

**TEAM Members:** Please EDIT this cell and add the names of all the team members in your team

    Jake Dorick
    
    Calin Ocnean-Sanchez
    
    ...

**Don't forget!**
* You will need to install the pymongo library to access MongoDB
 * pip install pymongo
* You will also need to install dnspython to access the cloud version of MongoDB
 * pip install dnspython

** NOTE **
* **Please don't forget to save the notebook frequently when working in Jupyter notebook, otherwise the changes you made can be lost**.

# Problem 1 (20 points):  Get a cloud database account using MongoDB Atlas

Get a free cloud hosted MongoDB database account at https://www.mongodb.com/atlas/database.  You will need to:

* Create a database cluster using the MongoDB web interface
* Get your MongoDB credentials
* Create a document collection in MongoDB


*LLM-Citation:* Claude Code was used to generate and modify code for the cell below.

In [1]:
#----------------------------------------------
# Your code starts here
#   Please add comments or text cells in between to explain the general idea of each block of the code.
#   Please feel free to add more cells below this cell if necessary

import pymongo
import os
from dotenv import load_dotenv

# Get credentials from MongoDB URI stored in .env file
load_dotenv()
uri = os.getenv("MONGODB_URI")

client = pymongo.MongoClient(uri)

# Confirm connection to MongoDB by sending a ping command
client.admin.command("ping")
print("Connected to MongoDB.")

# Connect to the database and collection
db = client["NewsDatabase"]
collection = db["Articles"]

# Insert a test document into the collection
collection.insert_one({"name": "test item", "quantity": 1})
collection.insert_many([
    {"name": "apple", "quantity": 5},
    {"name": "banana", "quantity": 12},
])
print("Inserted documents.")

Connected to MongoDB.
Inserted documents.


![Test Document](Imgs/Test_Documents.png)

Worked.

# Problem 2 (20 points):  Read NewsAPI data into your cloud MongoDB database

Gather NewsAPI data and upload it to your cloud database.  You will need to:

* Get your cloud MongoDB credentials using the MongoDB web interface
* Insert those credentials into your Jupyter notebook using pymongo
* Gather NewsAPI data as in Case Study 1
* Upload your NewsAPI data to the MongoDB cloud document collection

Note, to do you this question remember that you will need *two sets of credentials*

* One set for the MongoDB Atlas database
* One set for the NewsAPI

Of course, there are not the same!

*LLM-Citation:* Claude Code was used to generate and modify code for the cells below.

In [7]:
import pymongo
from newsapi import NewsApiClient
import os
from dotenv import load_dotenv

# See https://newsapi.org/docs for more information
# on NewsAPI implementation.

load_dotenv(".env", override=True)

API_KEY = os.getenv("API_KEY")
MONGODB_URI = os.getenv("MONGODB_URI")

# Connect with pymongo and ping the server
client = pymongo.MongoClient(MONGODB_URI)
client.admin.command("ping")
print("Connected to MongoDB Atlas.")

# Connect to the database and collection
db = client["NewsDatabase"]
collection = db["Articles"]

Connected to MongoDB Atlas.


### Use the same method as in Case Study 1 to collect the articles from NewsAPI

In [8]:
# Search one day at a time over the past 30 days to work around the free-tier limit

from datetime import datetime, timedelta

newsapi = NewsApiClient(api_key=API_KEY)

end_date = datetime.now().date()
start_date = end_date - timedelta(days=29)

all_articles = []
seen_urls = set()
current_date = start_date

while current_date <= end_date:
    date_string = current_date.strftime("%Y-%m-%d")

    results = newsapi.get_everything(
        q='"Texas Senate"',
        from_param=date_string,
        to=date_string,
        language="en",
        sort_by="publishedAt",
        page_size=100,
    )

    articles = results["articles"]
    for article in articles:
        url = article.get("url")
        if url and url not in seen_urls:
            all_articles.append(article)
            seen_urls.add(url)

    print(f"{date_string}: {len(articles)} articles")
    current_date += timedelta(days=1)

print(f"\nTotal unique articles collected: {len(all_articles)}")

2026-10-23: 12 articles
2026-10-24: 6 articles
2026-10-25: 5 articles
2026-10-26: 4 articles
2026-10-27: 20 articles
2026-10-28: 21 articles
2026-10-29: 18 articles
2026-10-30: 24 articles
2026-10-31: 9 articles
2026-11-01: 9 articles
2026-11-02: 19 articles
2026-11-03: 27 articles
2026-11-04: 97 articles
2026-11-05: 34 articles
2026-11-06: 11 articles
2026-11-07: 10 articles
2026-11-08: 8 articles
2026-11-09: 5 articles
2026-11-10: 11 articles
2026-11-11: 7 articles
2026-11-12: 8 articles
2026-11-13: 6 articles
2026-11-14: 4 articles
2026-11-15: 1 articles
2026-11-16: 5 articles
2026-11-17: 2 articles
2026-11-18: 5 articles
2026-11-19: 13 articles
2026-11-20: 10 articles
2026-11-21: 0 articles

Total unique articles collected: 411


### Delete the test documents

In [ ]:
# Remove the test documents to prevent DuplicateKeyErrors
deleted = collection.delete_many({"url": {"$exists": False}})
print("Deleted:", deleted.deleted_count)

collection.create_index("url", unique=True)

Deleted: 3


'url_1'

### Finish by uploading the articles to the MongoDB collection.

In [12]:
from pymongo import ReplaceOne

collection.create_index("url", unique=True)

operations = [
    ReplaceOne({"url": article["url"]}, article, upsert=True)
    for article in all_articles
    if article.get("url")
]

result = collection.bulk_write(operations)
print(f"Inserted: {result.upserted_count}, updated: {result.modified_count}")

# Verify the number of documents and show a few sample documents
print("Documents in collection:", collection.count_documents({}))

for doc in collection.find({}, {"_id": 0, "title": 1, "source.name": 1, "publishedAt": 1}).limit(5):
    print(doc)

Inserted: 411, updated: 0
Documents in collection: 411
{'source': {'name': 'The Atlantic'}, 'title': 'The GOP’s Midterms-Money Advantage Has Its Limits', 'publishedAt': '2026-10-23T19:45:00Z'}
{'source': {'name': 'FactCheck.org'}, 'title': 'Paxton Ad Takes Talarico Taxes Comments Out of Context', 'publishedAt': '2026-10-23T19:01:56Z'}
{'source': {'name': 'Smallwarsjournal.com'}, 'title': '10/23/26 National Security and Korean News and Commentary', 'publishedAt': '2026-10-23T19:00:45Z'}
{'source': {'name': 'Wired'}, 'title': 'Elon Musk Just Dumped Another $76 Million Into the Midterms', 'publishedAt': '2026-10-23T16:12:49Z'}
{'source': {'name': 'Reason'}, 'title': 'Could a Libertarian Swing the Texas Senate Race?', 'publishedAt': '2026-10-23T15:00:49Z'}


# Problem 3 (20 points):  Use a regular expression to read a subset of your news articles out of MongoDB

Use regular expressions to analyze properties of your data.  For example, you can

* search for descriptions that contain a certain string,
* search for articles that have images
* search for certain news sources
* anything else you think might be useful for a product!

In [ ]:
#----------------------------------------------
# Your code starts here
#   Please add comments or text cells in between to explain the general idea of each block of the code.
#   Please feel free to add more cells below this cell if necessary


# Problem 4 (20 points): Business question

Build on the product you selected in Case Study 1 to explore how MongoDB could support its functionality and growth.

* Develop a business question related to your product that NewsAPI data and MongoDB could help answer.
* Describe the business case, including the target users and how your product would benefit them.
* **[LLM only]**: Ask your favorite LLM to suggest three ways MongoDB could support your product. Describe your product and the NewsAPI data you collected in your question, and include your question and the LLM’s response in your report.
* **[Human only]**: Evaluate the three suggestions and select the most useful one for your product. Explain why you selected it and how you would use MongoDB to implement it.
* **[LLM only]**: For the same product, ask the LLM to identify three potential scaling challenges as the data volume, number of users, or query frequency increases. Include your question and the LLM’s response in your report.
* **[Human only]**: Evaluate the three scaling challenges and identify the most relevant one for your product. Explain why it matters and propose an approach to address it using MongoDB.

In [ ]:
#----------------------------------------------
# Your code starts here
#   Please add comments or text cells in between to explain the general idea of each block of the code.
#   Please feel free to add more cells below this cell if necessary





*-----------------
# Done

All set!

** What do you need to submit?**

* **Notebook File**: Save this Jupyter notebook, and find the notebook file in your folder (for example, "filename.ipynb"). This is the file you need to submit. Please make sure all the plotted tables and figures are in the notebook All the figures and tables should have shown up in the notebook.

* **Report**: please prepare a report (less than 10 pages) to describe your data, findings, and how MongoDB supports your product. **The report should explain what motivated the work and cover every step outlined in the case study.** In addition, address the following questions:

    * What data did you collect, and why is it relevant to your product?
    * Who are your target users, and why is the topic important to them?
    * How did you store, query, and analyze your data using MongoDB?
    * What did you find, and how do your findings support your product?
    * What benefits does MongoDB provide compared with storing your data in local files, such as CSV or JSON? Discuss specific examples from your work and any limitations or tradeoffs.
    * How could your database approach support growth in data volume and number of users?

     (please include relevant figures or tables, your LLM questions and responses, and your own evaluations. Do not include source code.)

* **Video**: A 2–3-minute video pitch to a venture capitalist introducing your product, its target users, and the value it provides. **The video should tell a clear story that connects what motivated the work, what the results show, and why those results matter.** In addition, briefly describe your data and key findings, and explain why a cloud database is useful for your product. Use a concrete example from your work to illustrate how MongoDB supports a product feature, and discuss how it could help your product grow. Include a preview of your product. **Submit only a link to your video, not the video file. Include the link to your video in your report file. You may upload your video to Google Drive, OneDrive, YouTube (Unlisted) or any other video-sharing platform. Ensure that anyone with the shared link can view the video without requesting access.**

Please compress the notebook file and the report (including the video link) into a zipped file and submit it on canvas.wpi.edu for grading.



# Grading Criteria:

**Total Points: 100**


---------------------------------------------------------------------------
**Notebook results:**
    Points: 80


    -----------------------------------
    Question 1:
    Points: 20
    -----------------------------------
    
    -----------------------------------
    Question 2:
    Points: 20
    -----------------------------------
        
    -----------------------------------
    Question 3:
    Points: 20
    -----------------------------------
  
    -----------------------------------
    Question 4:  Business question
    Points: 20
    -----------------------------------

  
  **Report and video:**
    Points: 20
